# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/georgy-com/Flyrank-Repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I will frame my lane as a **classification** problem. The goal is to classify each page as declining or not declining using observable page and search-performance signals such as content age, days since the last update, impressions, average position, CTR, word count, and engagement rate. Classification fits this decision because the practical outcome is to identify pages that should be prioritized for review rather than simply estimate a continuous value. The model will be used as decision-support for content prioritization.


In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))
# Confirm the target classes available in the dataset
print("Target classes:")
print(df["is_declining_label"].value_counts())

print("\nTarget proportions:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

30000 pages |  declining rate: 0.542
Target classes:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target proportions:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is **`is_declining_label`**, where 1 represents a page classified as declining and 0 represents a page that is not classified as declining. The label is derived from the observed trend information in the starter dataset, rather than being a manually created business opinion. Therefore, it is best treated as a defined outcome label for this exercise.

The model will use pre-decision signals such as `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`, and `engagement_rate`. I will not use `trend_direction` or `trend_pct` as features because they are directly related to how the target is defined and would create leakage.


In [3]:
# Check how the target is distributed
target_counts = df["is_declining_label"].value_counts().sort_index()

print("Target distribution:")
print(target_counts)

print("\nTarget rate:")
print(
    f"Declining rate: "
    f"{df['is_declining_label'].mean():.3f}"
)

Target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Target rate:
Declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric will be **Precision@50**. This measures the proportion of the 50 highest-ranked pages that are actually classified as declining. I chose it because the practical use case is prioritization: a content or SEO team has limited time and needs a short list of pages to review first.

A higher Precision@50 means that more of the pages prioritized by the model are actually declining. I will compare the model against the hand-rule baseline and, where possible, evaluate it on held-out data rather than relying only on in-sample performance.


In [4]:
# Precision@K helper used throughout the Week-2 experiment

import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Primary metric: Precision@50")

Primary metric: Precision@50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one content/page performance record**. Each row represents one anonymized piece of content with its associated search-performance and content characteristics. The row contains variables such as impressions, clicks, CTR, average position, content age, update recency, engagement, and the observed trend outcome.

For this ML task, one row therefore represents **one page/content record that the model will classify as declining or not declining**.


In [5]:
# Show the structure of one observation
print("Dataset shape:")
print(df.shape)

print("\nExample row:")
display(
    df[
        [
            "content_id",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "engagement_rate",
            "is_declining_label"
        ]
    ].head(5)
)

print("\nColumns used for the ML lane:")
lane_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate",
    "is_declining_label"
]

print(lane_columns)

Dataset shape:
(30000, 45)

Example row:


,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,engagement_rate,is_declining_label
0,content_304f48230142,187,20,3803,10.6,0.76,5.88,1
1,content_a1fb4e703a9e,445,25,15320,20.3,0.05,0.00,1
2,content_9aa793d4d895,141,20,12581,36.5,0.09,0.00,1
3,content_331d6c4de07b,463,22,11751,6.2,0.49,1.28,0
4,content_d99b7a2d90ca,263,14,19140,44.0,0.13,0.00,1



Columns used for the ML lane:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'engagement_rate', 'is_declining_label']


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule is useful as a simple baseline, but the observed page-performance patterns are more complicated than a single if-statement. A page can be old and highly visible without declining, while another page may be relatively recent but show weaker performance signals. Different combinations of impressions, position, CTR, content age, update recency, word count, and engagement may correspond to different outcomes.

The decision tree experiment showed that a model can combine multiple signals into readable if/else rules rather than relying on one fixed threshold. This makes ML useful when the relationship between the available signals and declining status is too complex for a single hand-written rule. The model will still be evaluated against the simple hand rule to check whether the additional complexity provides useful decision-support.


In [6]:

hand_rule_features = [
    "days_since_last_update",
    "impressions_90d"
]

model_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

print("Hand-rule signals:")
print(hand_rule_features)

print("\nCandidate ML signals:")
print(model_features)

print(
    f"\nHand rule uses {len(hand_rule_features)} signals."
)

print(
    f"Candidate ML lane uses {len(model_features)} signals."
)

Hand-rule signals:
['days_since_last_update', 'impressions_90d']

Candidate ML signals:
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count', 'engagement_rate']

Hand rule uses 2 signals.
Candidate ML lane uses 7 signals.
